In [1]:
from pathlib import Path

DATA_RAW = Path("datasets/raw")
DATA_PROCESSED = Path("datasets/processed")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
SPLITS = ["train", "val", "test"]
# SPLITS = ["train", "test"]


In [2]:
import shutil

#delete processed folder
if DATA_PROCESSED.exists():
    shutil.rmtree(DATA_PROCESSED)

In [3]:
#delete duplicates /near duplicates
    #pHash distance or CLIP embeddings

#Annotation Checks
    # every value between 0 and 1 

#Minimum Box Size
    #w adn h > 0.005


## scan for duplicates

In [4]:
from pathlib import Path
from PIL import Image
import imagehash

hashes = {}
duplicates = 0

for split in SPLITS:

    image_dir = DATA_RAW / IMAGES_DIR / split

    for img_path in image_dir.glob("*.jpg"):

        try:
            h = imagehash.phash(
                Image.open(img_path)
            )

            if h in hashes:
                duplicates += 1
                print(
                    "Duplicate found:",
                    img_path,
                    "<->",
                    hashes[h]
                )

            hashes[h] = img_path

        except Exception as e:

            print(
                "Failed:",
                img_path,
                e
            )

print("Done - total duplicates:", duplicates)

Duplicate found: datasets\raw\images\train\0_8083.jpg <-> datasets\raw\images\train\0_8082.jpg
Duplicate found: datasets\raw\images\train\0_8084.jpg <-> datasets\raw\images\train\0_8083.jpg
Duplicate found: datasets\raw\images\train\0_8087.jpg <-> datasets\raw\images\train\0_8086.jpg
Duplicate found: datasets\raw\images\train\0_8090.jpg <-> datasets\raw\images\train\0_8089.jpg
Duplicate found: datasets\raw\images\train\0_8091.jpg <-> datasets\raw\images\train\0_8090.jpg
Duplicate found: datasets\raw\images\train\0_8093.jpg <-> datasets\raw\images\train\0_8092.jpg
Duplicate found: datasets\raw\images\train\0_8097.jpg <-> datasets\raw\images\train\0_8096.jpg
Duplicate found: datasets\raw\images\train\0_8098.jpg <-> datasets\raw\images\train\0_8097.jpg
Duplicate found: datasets\raw\images\train\0_8101.jpg <-> datasets\raw\images\train\0_8099.jpg
Duplicate found: datasets\raw\images\train\0_8103.jpg <-> datasets\raw\images\train\0_8102.jpg
Duplicate found: datasets\raw\images\train\0_8105.

### check labels, black, blurry, thermal normalization, validate labels, save celaned

In [5]:
#delete black and blurry images
import cv2
import numpy as np
from collections import defaultdict

BLACK_PIXEL_THRESHOLD = 0.90  # 93% pixels near black
BLUR_THRESHOLD = 50

stats = defaultdict(
    lambda: {
        "total": 0,
        "kept": 0,
        "black": 0,
        "blurry": 0,
        "missing_label": 0,
        "invalid_label": 0,
        "corrupt": 0,
    }
)


In [6]:
from tqdm import tqdm
from utils.preprocess_methods import normalize_thermal, validate_label_line, is_mostly_black, is_blurry

for split in SPLITS:

    image_src = DATA_RAW / IMAGES_DIR / split
    label_src = DATA_RAW / LABELS_DIR / split

    image_dst = DATA_PROCESSED / IMAGES_DIR / split
    label_dst = DATA_PROCESSED / LABELS_DIR / split

    image_dst.mkdir(parents=True, exist_ok=True)
    label_dst.mkdir(parents=True, exist_ok=True)

    image_files = list(image_src.glob("*.jpg"))

    for image_path in tqdm(image_files, desc=split):

        stats[split]["total"] += 1

        label_path = label_src / f"{image_path.stem}.txt"

        # --------------------------
        # label exists?
        # --------------------------
        if not label_path.exists():
            stats[split]["missing_label"] += 1
            print(f"[{split}] Missing label: {image_path.name}")
            continue

        # --------------------------
        # black image check
        # --------------------------
        if is_mostly_black(image_path):
            stats[split]["black"] += 1
            continue

        # --------------------------
        # blurry image check
        # --------------------------
        if is_blurry(image_path):
            stats[split]["blurry"] += 1
            continue

        # --------------------------
        # load image
        # --------------------------
        img = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if img is None:
            stats[split]["corrupt"] += 1
            continue

        # --------------------------
        # thermal normalization
        # --------------------------
        img = normalize_thermal(img)

        # --------------------------
        # validate labels
        # --------------------------
        valid_lines = []

        with open(label_path) as f:

            for line in f:

                parts = line.strip().split()

                if validate_label_line(parts):
                    valid_lines.append(line)

        if len(valid_lines) == 0:
            stats[split]["invalid_label"] += 1
            continue

        # --------------------------
        # save processed image
        # --------------------------
        cv2.imwrite(
            str(image_dst / image_path.name),
            img
        )

        # --------------------------
        # save cleaned labels
        # --------------------------
        with open(label_dst / label_path.name, "w") as f:
            f.writelines(valid_lines)

        stats[split]["kept"] += 1


# =====================================================
# SUMMARY
# =====================================================

print("\n===== PREPROCESSING SUMMARY =====")

for split in SPLITS:

    s = stats[split]

    print(f"\n[{split}]")
    print(f"Total Images   : {s['total']}")
    print(f"Kept           : {s['kept']}")
    print(f"Black Removed  : {s['black']}")
    print(f"Blur Removed   : {s['blurry']}")
    print(f"Corrupt Images : {s['corrupt']}")
    print(f"Missing Labels : {s['missing_label']}")
    print(f"Invalid Labels : {s['invalid_label']}")

train:  24%|██▍       | 2471/10136 [00:37<02:43, 46.78it/s]

Very small box detected (w=0.0049, h=0.0439)
Very small box detected (w=0.0078, h=0.0371)


train:  25%|██▍       | 2486/10136 [00:37<02:49, 45.05it/s]

Very small box detected (w=0.0635, h=0.0088)
Very small box detected (w=0.0312, h=0.0059)


train:  26%|██▋       | 2671/10136 [00:41<02:07, 58.74it/s]

...

Very small box detected (w=0.0166, h=0.0029)
Very small box detected (w=0.0156, h=0.0059)
Very small box detected (w=0.0176, h=0.0068)
Very small box detected (w=0.0225, h=0.0068)
Very small box detected (w=0.0234, h=0.0059)
Very small box detected (w=0.0254, h=0.0059)


test:  96%|█████████▌| 1791/1864 [00:39<00:01, 41.26it/s]

Very small box detected (w=0.0010, h=0.0322)


test: 100%|██████████| 1864/1864 [00:40<00:00, 45.54it/s]

Very small box detected (w=0.0439, h=0.0049)

===== PREPROCESSING SUMMARY =====

[train]
Total Images   : 10136
Kept           : 5240
Black Removed  : 0
Blur Removed   : 4896
Corrupt Images : 0
Missing Labels : 0
Invalid Labels : 0

[val]
Total Images   : 84
Kept           : 84
Black Removed  : 0
Blur Removed   : 0
Corrupt Images : 0
Missing Labels : 0
Invalid Labels : 0

[test]
Total Images   : 1864
Kept           : 1642
Black Removed  : 0
Blur Removed   : 222
Corrupt Images : 0
Missing Labels : 0
Invalid Labels : 0


In [7]:
print("PreProcessing completed - stats:")

def count_images(base_path):
    counts = {}
    for split in SPLITS:
        path = base_path / IMAGES_DIR / split
        counts[split] = len(list(path.glob("*.jpg")))
    return counts


raw_counts = count_images(DATA_RAW)
processed_counts = count_images(DATA_PROCESSED)



for split in SPLITS:
    before = raw_counts.get(split, 0)
    after = processed_counts.get(split, 0)
    print(f"{split.upper():5s} | before: {before:5d} | after: {after:5d}")

PreProcessing completed - stats:
TRAIN | before: 10136 | after:  5240
VAL   | before:    84 | after:    84
TEST  | before:  1864 | after:  1642


In [8]:
# filling missing val data:

In [9]:
from pathlib import Path
import shutil

# Flight IDs to move into validation set
train_flights = {"129", "155", "211", "305"}
test_flights = {"163", "263", "277", "284", "314", "343"}

dataset_root = DATA_PROCESSED

def copy_matching_files(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, flight_ids):
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_file in src_img_dir.glob("*.jpg"):
        flight_id = img_file.stem.split("_")[0]

        if flight_id in flight_ids:
            # Copy image
            shutil.copy2(img_file, dst_img_dir / img_file.name)

            # Copy corresponding label if it exists
            label_file = src_lbl_dir / f"{img_file.stem}.txt"
            if label_file.exists():
                shutil.copy2(label_file, dst_lbl_dir / label_file.name)

            print(f"Copied: {img_file.name}")

# From TRAIN -> VAL
copy_matching_files(
    dataset_root / "images" / "train",
    dataset_root / "labels" / "train",
    dataset_root / "images" / "val",
    dataset_root / "labels" / "val",
    train_flights,
)

# From TEST -> VAL
copy_matching_files(
    dataset_root / "images" / "test",
    dataset_root / "labels" / "test",
    dataset_root / "images" / "val",
    dataset_root / "labels" / "val",
    test_flights,
)

print("Done.")

Copied: 129_1681.jpg
Copied: 129_1691.jpg
Copied: 129_1701.jpg
Copied: 129_1711.jpg
Copied: 129_1712.jpg
Copied: 129_1715.jpg
Copied: 129_1716.jpg
Copied: 129_1725.jpg
Copied: 129_1726.jpg
Copied: 129_1735.jpg
Copied: 129_1745.jpg
Copied: 129_1758.jpg
Copied: 129_1762.jpg
Copied: 129_1768.jpg
Copied: 129_1772.jpg
Copied: 129_1775.jpg
Copied: 129_1785.jpg
Copied: 129_1795.jpg
Copied: 129_1802.jpg
Copied: 129_1825.jpg
Copied: 129_1838.jpg
Copied: 129_1863.jpg
Copied: 129_1875.jpg
Copied: 129_1876.jpg
Copied: 129_1895.jpg
Copied: 129_1905.jpg
Copied: 129_1915.jpg
Copied: 129_1925.jpg
Copied: 129_1935.jpg
Copied: 129_1945.jpg
Copied: 129_1955.jpg
Copied: 129_1975.jpg
Copied: 129_1987.jpg
Copied: 129_1997.jpg
Copied: 129_2007.jpg
Copied: 129_3200.jpg
Copied: 129_3220.jpg
Copied: 129_3230.jpg
Copied: 129_3240.jpg
Copied: 129_3250.jpg
Copied: 129_3270.jpg
Copied: 129_3273.jpg
Copied: 129_3280.jpg
Copied: 129_3282.jpg
Copied: 129_3288.jpg
Copied: 129_3290.jpg
Copied: 129_3300.jpg
Copied: 129_3

In [10]:
print("after filling missing val data:")

raw_counts = count_images(DATA_RAW)
processed_counts = count_images(DATA_PROCESSED)


for split in SPLITS:
    before = raw_counts.get(split, 0)
    after = processed_counts.get(split, 0)
    print(f"{split.upper():5s} | before: {before:5d} | after: {after:5d}")

after filling missing val data:
TRAIN | before: 10136 | after:  5240
VAL   | before:    84 | after:   657
TEST  | before:  1864 | after:  1642


In [11]:
## MORE: